# Feature selection — ordinal (proportional-odds) logistic regression
**Virginia tobacco land suitability, Regions I & II**

Run the cells in order. Each stage leaves its variables in memory, so a stage can be
re-run on its own without repeating the ones before it.

Agronomy decides what enters the candidate pool; statistics only adjudicate redundancy
and form. Priority-1 (decisive) factors are forced in and are never dropped on p-values.

| Stage | What it does | Output |
|---|---|---|
| 0 | Load, spatial holdout (stratified on S1-bearing blocks) | — |
| 1 | Transforms + search subsample | — |
| 2 | Univariate ordinal screen | `fs_univariate_screen.csv` |
| 3a | Spearman redundancy | log |
| 3b | Iterative VIF | `fs_vif_final.csv` |
| 4 | p-value screen (forced vars exempt) | log |
| 5 | Backward selection with confounding check | log |
| 6 | L1 stability (reported, not acted on) | `fs_stability.csv` |
| 7 | Proportional-odds check | `fs_proportional_odds_check.csv` |
| 8 | Final model + block bootstrap | `fs_final_odds_ratios.csv` |
| 9 | Spatially blocked CV + holdout by region | `fs_cv_comparison.csv` |

## Setup

In [1]:
# !pip install statsmodels scikit-learn pyarrow -q
import os, time, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 50)

DATA_PATH = r"D:\Projects\capstone\feature_matrix.parquet"
OUT_DIR   = r"D:\Projects\capstone\fs_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
p_out = lambda n: os.path.join(OUT_DIR, n)

TARGET, BLOCK, REGION = "fao_class", "spatial_block", "region"
ORDER = ["N", "S3", "S2", "S1"]      # low -> high; positive coef = more suitable
SEED  = 42

SELECT_N = 60_000     # rows for the SEARCH stages (2-7). 0 = use all.
N_BOOT   = 100        # block-bootstrap reps
BOOT_N   = 40_000     # rows per bootstrap rep

T0 = time.time()
log = []
def note(feat, stage, decision, reason):
    log.append(dict(feature=feat, stage=stage, decision=decision, reason=reason))
def tick(msg):
    print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("ready")

ready


### Candidate pool and encoding rules

`PRIORITY` is the agronomic ranking: 1 = decisive (forced in), 2 = core Table 3.1,
3 = supporting, 4 = supplementary.

`rain_dryseason` has **no** `OPTIMUM` entry on purpose — Table 3.1 grades *annual*
rainfall and this column is the Nov–Apr total, so no published optimum applies to it.

In [2]:
PRIORITY = {
    "soil_ph": 1, "slope_pct": 1, "temp_gs_mean": 1, "ndmi_median": 1, "twi": 1,

    "sand_pct": 2, "clay_pct": 2, "soc": 2, "total_n": 2, "cec": 2,
    "sunshine_rad": 2, "rain_dryseason": 2,

    "elevation": 3, "aspect": 3, "vpd_med": 3, "ndvi_median": 3,
    "water_freq": 3, "awc": 3, "tpi": 3, "soil_moist_med": 3,
    "dry_spell_max_med": 3, "precip_cv_med": 3,

    "ndvi_med_iqr": 4, "ndmi_med_iqr": 4, "ndmi_range_med": 4,
    "stress_freq_med": 4, "amplitude_med": 4, "dss_peak_med": 4,
    "tmean_iqr": 4, "precip_total_iqr": 4, "solar_rad_iqr": 4,
}

CATEGORICAL = []                 # no land-cover layer was produced
CIRCULAR    = ["aspect"]         # reconstructed from northness/eastness

# EDA skewness table, |skew| >= 1.0, restricted to columns present here
SKEWED = ["soc", "total_n", "water_freq", "ndvi_med_iqr", "ndmi_med_iqr",
          "ndmi_range_med", "stress_freq_med", "elevation", "slope_pct",
          "twi", "amplitude_med"]

# centred on the Table 3.1 S1 midpoint + squared term; None = training median
OPTIMUM = {"soil_ph": 5.85, "temp_gs_mean": 25.0, "total_n": None,
           "cec": 18.5, "soc": None}

CORR_CUT, VIF_CUT = 0.70, 10.0
SCREEN_P, KEEP_P, CONFOUND = 0.25, 0.10, 0.20
print(f"{len(PRIORITY)} candidate variables")

31 candidate variables


## Stage 0 — Load and spatial holdout

The holdout is taken **before any selection**, and block sampling is stratified on
whether a block contains S1: only ~50 of ~1,000 blocks do, so an unstratified draw
could leave the holdout with none.

**Check:** the holdout class counts must include S1.

In [3]:
tick("loading")
df = pd.read_parquet(DATA_PATH).dropna(subset=[TARGET]).reset_index(drop=True)
PRIORITY = {k: v for k, v in PRIORITY.items() if k in df.columns}

rng = np.random.default_rng(SEED)
test_blocks = []
for _, g in df.groupby(REGION):
    has_rare = g.groupby(BLOCK)[TARGET].apply(lambda s: (s == ORDER[-1]).any())
    for flag in (True, False):
        b = has_rare[has_rare == flag].index.to_numpy()
        if len(b):
            test_blocks += list(rng.choice(b, size=max(1, int(0.2*len(b))), replace=False))

is_test = df[BLOCK].isin(test_blocks)
train = df[~is_test].reset_index(drop=True)
test  = df[is_test].reset_index(drop=True)
to_y  = lambda s: pd.Series(pd.Categorical(s, categories=ORDER, ordered=True))
y_tr_full, y_te = to_y(train[TARGET]), to_y(test[TARGET])

print(f"rows {len(df):,} | candidates {len(PRIORITY)}")
print(f"train {len(train):,} | holdout {len(test):,} over {len(test_blocks)} blocks")
pd.DataFrame({"all": df[TARGET].value_counts().reindex(ORDER),
              "train": train[TARGET].value_counts().reindex(ORDER),
              "holdout": test[TARGET].value_counts().reindex(ORDER)})

[   0.4 min] loading
rows 375,122 | candidates 31
train 303,096 | holdout 72,026 over 105 blocks


,all,train,holdout
fao_class,,,
N,120000,96141,23859
S3,120000,95705,24295
S2,120000,98014,21986
S1,15122,13236,1886


## Stage 1 — Transforms, then the search subsample

Skewed variables get `log1p`, aspect becomes sin/cos, and optimum-band variables are
centred on their Table 3.1 midpoint with a squared term so the model can express
"too low *and* too high are both worse". All scaling is fitted on **train only**.

Backward selection fits O(k²) ordinal models, which at 300k rows runs for hours, so the
*search* uses a class-stratified 60k subsample. The final model in Stage 8 and the
holdout evaluation in Stage 9 use the full training set.

In [4]:
def build_features(d, ref):
    X, groups = pd.DataFrame(index=d.index), {}
    for v in PRIORITY:
        if v in CATEGORICAL:
            levels = ref[v].value_counts().index            # most frequent = reference
            cols = [f"{v}_{lv}" for lv in levels[1:]]
            for lv, c in zip(levels[1:], cols):
                X[c] = (d[v] == lv).astype(float)
            groups[v] = cols
        elif v in CIRCULAR:
            r = np.deg2rad(d[v]); X[f"{v}_north"], X[f"{v}_east"] = np.cos(r), np.sin(r)
            groups[v] = [f"{v}_north", f"{v}_east"]
        elif v in OPTIMUM:
            c = OPTIMUM[v] if OPTIMUM[v] is not None else ref[v].median()
            z = (d[v] - c) / ref[v].std()
            X[v], X[f"{v}_sq"] = z, z**2
            groups[v] = [v, f"{v}_sq"]
        else:
            f = (lambda s: np.log1p(s.clip(lower=0))) if v in SKEWED else (lambda s: s.astype(float))
            x, xr = f(d[v]), f(ref[v])
            X[v] = (x - xr.mean()) / xr.std()
            groups[v] = [v]
    return X, groups

X_tr_full, GROUPS = build_features(train, train)
X_te, _ = build_features(test, train)
for v in PRIORITY:
    note(v, "1_transform", "kept",
         "one-hot" if v in CATEGORICAL else "sin/cos" if v in CIRCULAR else
         "centred on optimum + squared" if v in OPTIMUM else
         "log1p + z-score" if v in SKEWED else "z-score")

if SELECT_N and len(train) > SELECT_N:
    take, per_cls = [], SELECT_N // len(ORDER)
    for c in ORDER:
        idx = np.flatnonzero((train[TARGET] == c).to_numpy())
        take.append(rng.choice(idx, min(len(idx), per_cls), replace=False))
    sel = np.sort(np.concatenate(take))
    X_sel  = X_tr_full.iloc[sel].reset_index(drop=True)
    y_sel  = y_tr_full.iloc[sel].reset_index(drop=True)
    blk_sel = train[BLOCK].to_numpy()[sel]
else:
    X_sel, y_sel, blk_sel = X_tr_full, y_tr_full, train[BLOCK].to_numpy()

print(f"design matrix: {X_tr_full.shape[1]} columns from {len(GROUPS)} variables")
print(f"search subsample: {len(X_sel):,} rows")
pd.Series(y_sel).value_counts().reindex(ORDER).rename("search subsample")

design matrix: 37 columns from 31 variables
search subsample: 58,236 rows


N     15000
S3    15000
S2    15000
S1    13236
Name: search subsample, dtype: int64

## Helpers

In [5]:
def fit_ord(X, y):
    return OrderedModel(y, X, distr="logit").fit(method="bfgs", maxiter=3000, disp=False)

def null_llf(y):
    n = pd.Series(y).value_counts(); n = n[n > 0]
    return float((n * np.log(n / n.sum())).sum())

def cols_of(groups):
    return [c for g in groups for c in GROUPS[g]]
print("ok")

ok


## Stage 2 — Univariate ordinal screen

Each variable alone against the null model, by likelihood ratio. Feeds the tie-breaks
in Stage 3a and the p-value screen in Stage 4.

In [6]:
tick("stage 2")
LL0, uni = null_llf(y_sel), []
for v, cols in GROUPS.items():
    try:
        m  = fit_ord(X_sel[cols], y_sel)
        lr = 2 * (m.llf - LL0)
        uni.append(dict(feature=v, priority=PRIORITY[v], LR_chi2=lr, df=len(cols),
                        p=stats.chi2.sf(lr, len(cols))))
    except Exception as e:
        uni.append(dict(feature=v, priority=PRIORITY[v], LR_chi2=np.nan, df=len(cols), p=np.nan))
        note(v, "2_univariate", "warning", f"fit failed: {e}")
uni = pd.DataFrame(uni).set_index("feature")
uni.sort_values("LR_chi2", ascending=False).to_csv(p_out("fs_univariate_screen.csv"))
active = list(GROUPS)
uni.sort_values("LR_chi2", ascending=False).round(4)

[   1.6 min] stage 2


,priority,LR_chi2,df,p
feature,,,,
slope_pct,1,65576.0983,1,0.0000
sand_pct,2,38471.8785,1,0.0000
clay_pct,2,38315.0298,1,0.0000
ndmi_median,1,30227.4663,1,0.0000
stress_freq_med,4,26592.0251,1,0.0000
awc,3,25983.7153,1,0.0000
soc,2,20805.4763,2,0.0000
ndvi_median,3,20451.0967,1,0.0000
temp_gs_mean,1,18967.7878,2,0.0000


## Stage 3a — Pairwise redundancy (Spearman)

Pairs above |ρ| 0.70. The lower-priority variable goes; on a tie, the weaker univariate
signal goes. **Two decisive factors that are collinear are kept and flagged**, not
silently dropped — that is the 3.4.3a rule.

In [7]:
tick("stage 3a")
num = [v for v in active if v not in CATEGORICAL + CIRCULAR]
rho = train[num].iloc[:200_000].corr(method="spearman")
pairs = [(a, b, rho.loc[a, b]) for i, a in enumerate(num) for b in num[i+1:]
         if abs(rho.loc[a, b]) > CORR_CUT]
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    if a not in active or b not in active: continue
    pa, pb = PRIORITY[a], PRIORITY[b]
    if pa == pb == 1:
        note(f"{a} ~ {b}", "3a_correlation", "flag",
             f"rho={r:.2f}, both decisive -> both kept, review"); continue
    if pa != pb:
        drop, why = (a if pa > pb else b), "lower agronomic priority"
    else:
        drop = a if uni.loc[a,"LR_chi2"] < uni.loc[b,"LR_chi2"] else b
        why  = "same priority, weaker univariate signal"
    keep = b if drop == a else a
    active.remove(drop)
    note(drop, "3a_correlation", "dropped", f"rho={r:.2f} with {keep}; {why}")

print(f"{len(pairs)} pairs above |rho| {CORR_CUT}; {len(active)} variables remain")
pd.DataFrame([r for r in log if r["stage"] == "3a_correlation"])

[   3.1 min] stage 3a
21 pairs above |rho| 0.7; 22 variables remain


,feature,stage,decision,reason
0,stress_freq_med,3a_correlation,dropped,rho=-0.95 with ndmi_median; lower agronomic pr...
1,precip_total_iqr,3a_correlation,dropped,rho=0.90 with rain_dryseason; lower agronomic ...
2,ndvi_median,3a_correlation,dropped,rho=0.88 with ndmi_median; lower agronomic pri...
3,vpd_med,3a_correlation,dropped,rho=0.87 with temp_gs_mean; lower agronomic pr...
4,amplitude_med,3a_correlation,dropped,"rho=0.86 with ndmi_range_med; same priority, w..."
5,precip_cv_med,3a_correlation,dropped,rho=-0.85 with rain_dryseason; lower agronomic...
6,sunshine_rad,3a_correlation,dropped,"rho=-0.80 with rain_dryseason; same priority, ..."
7,total_n,3a_correlation,dropped,rho=-0.80 with temp_gs_mean; lower agronomic p...
8,dry_spell_max_med,3a_correlation,dropped,rho=-0.79 with rain_dryseason; lower agronomic...


## Stage 3b — Iterative VIF

Drops the worst variable above VIF 10 and recomputes, but never a priority-1 factor.
Squared terms are excluded: their collinearity with the linear term is structural.

In [8]:
tick("stage 3b")
def vif_series(groups):
    cols = [c for c in cols_of(groups) if not c.endswith("_sq")]
    Xc = sm.add_constant(X_sel[cols])
    return pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                     index=cols)

while True:
    vif  = vif_series(active)
    over = vif[vif > VIF_CUT]
    if over.empty: break
    col_to_group = {c: g for g in active for c in GROUPS[g]}
    cands = {col_to_group[c]: over[c] for c in over.index}
    droppable = {g: s for g, s in cands.items() if PRIORITY[g] > 1}
    if not droppable:
        note(", ".join(cands), "3b_VIF", "flag", "VIF>10 among decisive factors only -> kept, review")
        break
    drop = max(droppable, key=lambda g: (PRIORITY[g], droppable[g]))
    active.remove(drop)
    note(drop, "3b_VIF", "dropped", f"VIF={droppable[drop]:.1f}")

v = vif_series(active).rename("VIF").sort_values(ascending=False)
v.to_csv(p_out("fs_vif_final.csv"))
v.round(2)

[   3.2 min] stage 3b


sand_pct          5.59
clay_pct          5.25
temp_gs_mean      4.08
tmean_iqr         3.46
rain_dryseason    3.31
awc               3.19
slope_pct         3.14
elevation         2.79
ndmi_median       2.72
twi               2.06
soil_moist_med    2.01
soc               1.95
ndmi_range_med    1.94
soil_ph           1.86
ndvi_med_iqr      1.71
solar_rad_iqr     1.69
ndmi_med_iqr      1.64
dss_peak_med      1.47
tpi               1.34
cec               1.32
water_freq        1.07
aspect_east       1.01
aspect_north      1.00
Name: VIF, dtype: float64

## Stage 4 — Univariate p-value screen (decisive factors exempt)

In [9]:
for v in list(active):
    p = uni.loc[v, "p"]
    if PRIORITY[v] == 1:
        note(v, "4_screen", "forced in", f"decisive factor (p={p:.3g})")
    elif not (p < SCREEN_P):
        active.remove(v)
        note(v, "4_screen", "dropped", f"univariate p={p:.3g} >= {SCREEN_P}")
print(f"{len(active)} variables into backward selection:")
print(sorted(active))

22 variables into backward selection:
['aspect', 'awc', 'cec', 'clay_pct', 'dss_peak_med', 'elevation', 'ndmi_med_iqr', 'ndmi_median', 'ndmi_range_med', 'ndvi_med_iqr', 'rain_dryseason', 'sand_pct', 'slope_pct', 'soc', 'soil_moist_med', 'soil_ph', 'solar_rad_iqr', 'temp_gs_mean', 'tmean_iqr', 'tpi', 'twi', 'water_freq']


## Stage 5 — Backward purposeful selection

The slow stage, and where the methodology lives. A variable with p > 0.10 is dropped
**unless** removing it shifts a decisive factor's coefficient by more than 20%, in which
case it is retained as a confounder.

In [10]:
tick(f"stage 5: {len(active)} variables")
forced    = [v for v in active if PRIORITY[v] == 1]
protected = set(forced)
# for optimum-band vars the effect lives in the curvature term; the centred
# linear term sits near zero by design, so relative change on it is meaningless
full_main_cols = [GROUPS[v][-1] if v in OPTIMUM else GROUPS[v][0] for v in forced]

while True:
    full, tests = fit_ord(X_sel[cols_of(active)], y_sel), []
    for v in active:
        if v in protected: continue
        red = fit_ord(X_sel[cols_of([g for g in active if g != v])], y_sel)
        tests.append((v, stats.chi2.sf(2*(full.llf - red.llf), len(GROUPS[v])), red))
    if not tests: break
    v, p, red = max(tests, key=lambda t: t[1])
    if p <= KEEP_P: break
    chg   = {c: abs(red.params[c] - full.params[c]) / max(abs(full.params[c]), 0.1)
             for c in full_main_cols}
    worst = max(chg, key=chg.get)
    if chg[worst] > CONFOUND:
        protected.add(v)
        note(v, "5_multivariable", "kept (confounder)",
             f"p={p:.3g} but removal shifts {worst} by {chg[worst]:.0%}")
        tick(f"  kept {v} as confounder")
    else:
        active.remove(v)
        note(v, "5_multivariable", "dropped", f"LR p={p:.3g} > {KEEP_P}")
        tick(f"  dropped {v} (p={p:.3g}); {len(active)} remain")

for v in active:
    note(v, "5_multivariable", "final", f"priority {PRIORITY[v]}")
FINAL_COLS = cols_of(active)
print("\nFINAL VARIABLES:")
pd.DataFrame({"variable": active, "priority": [PRIORITY[v] for v in active]}
             ).sort_values(["priority", "variable"])

[   3.2 min] stage 5: 22 variables
[  20.4 min]   dropped dss_peak_med (p=0.61); 21 remain
[  36.3 min]   dropped ndvi_med_iqr (p=0.285); 20 remain
[  50.6 min]   dropped ndmi_med_iqr (p=0.213); 19 remain
[  63.4 min]   dropped tmean_iqr (p=0.12); 18 remain

FINAL VARIABLES:


,variable,priority
3,ndmi_median,1
1,slope_pct,1
0,soil_ph,1
2,temp_gs_mean,1
4,twi,1
8,cec,2
6,clay_pct,2
9,rain_dryseason,2
5,sand_pct,2
7,soc,2


## Stage 6 — L1 stability (reported, not acted on)

How often each variable survives an L1 fit on half the blocks. A decisive factor with a
low selection frequency is worth a sentence in the discussion, not a removal.

In [11]:
tick("stage 6")
blocks_sel, freq = pd.Series(blk_sel), pd.Series(0.0, index=list(GROUPS))
ublocks, cand_cols, REPS = blocks_sel.unique(), cols_of(GROUPS), 50
for _ in range(REPS):
    keep_b = rng.choice(ublocks, size=max(2, len(ublocks)//2), replace=False)
    sub = blocks_sel.isin(keep_b).to_numpy()
    if pd.Series(y_sel[sub]).nunique() < 2: continue
    l1 = LogisticRegression(penalty="l1", solver="saga", C=0.1, max_iter=3000,
                            class_weight="balanced")
    l1.fit(X_sel.loc[sub, cand_cols], y_sel[sub].astype(str))
    nz = pd.Series(np.abs(l1.coef_).max(axis=0) > 1e-6, index=cand_cols)
    for g in GROUPS: freq[g] += nz[GROUPS[g]].any()

s = (freq/REPS).rename("L1_selection_freq").sort_values(ascending=False)
s.to_csv(p_out("fs_stability.csv"))
s.to_frame().assign(priority=lambda d: [PRIORITY[i] for i in d.index],
                    in_final=lambda d: [i in active for i in d.index])

[  74.4 min] stage 6


,L1_selection_freq,priority,in_final
soil_ph,1.00,1,True
slope_pct,1.00,1,True
temp_gs_mean,1.00,1,True
ndmi_median,1.00,1,True
twi,1.00,1,True
sand_pct,1.00,2,True
soc,1.00,2,True
rain_dryseason,1.00,2,True
total_n,1.00,2,False
cec,1.00,2,True


## Stage 7 — Proportional-odds check

Separate binary logits per cumulative split, testing whether each slope is equal across
splits. This is an approximation (it ignores cross-split covariance); for the formal
Brant test use R's `brant::brant()`.

If most terms show `PO_violated = True`, the proportional-odds assumption fails and a
partial-proportional-odds model is indicated — say so rather than ignoring it.

In [12]:
tick("stage 7")
split_b, split_se = {}, {}
codes = pd.Series(y_sel).cat.codes
for k in range(1, len(ORDER)):
    yb = (codes >= k).astype(int)
    try:
        r = sm.Logit(yb, sm.add_constant(X_sel[FINAL_COLS])).fit(disp=False, maxiter=200)
        split_b[ORDER[k]], split_se[ORDER[k]] = r.params[FINAL_COLS], r.bse[FINAL_COLS]
    except Exception as e:
        print(f"split >= {ORDER[k]} failed ({e}) -- likely separation")

po = []
if len(split_b) >= 2:
    B, SE = pd.DataFrame(split_b), pd.DataFrame(split_se)
    for c in FINAL_COLS:
        w    = 1 / SE.loc[c]**2
        bbar = (B.loc[c] * w).sum() / w.sum()
        chi  = (((B.loc[c] - bbar) / SE.loc[c])**2).sum()
        pv   = stats.chi2.sf(chi, B.shape[1]-1)
        po.append(dict(term=c, **{f"b_ge_{k}": B.loc[c, k] for k in B.columns},
                       chi2=chi, p=pv, PO_violated=pv < 0.05))
po = pd.DataFrame(po)
po.to_csv(p_out("fs_proportional_odds_check.csv"), index=False)
print(f"{int(po['PO_violated'].sum()) if len(po) else 0} of {len(po)} terms violate proportional odds")
po.round(4)

[ 122.4 min] stage 7
18 of 23 terms violate proportional odds


,term,b_ge_S3,b_ge_S2,b_ge_S1,chi2,p,PO_violated
0,soil_ph,-0.0793,0.7335,-0.2738,299.4536,0.0000,True
1,soil_ph_sq,-0.8999,-1.2132,-20.3935,327.6366,0.0000,True
2,slope_pct,-7.9909,-5.8289,-14.8116,358.9387,0.0000,True
3,temp_gs_mean,-0.6389,-0.0254,2.9849,61.6837,0.0000,True
4,temp_gs_mean_sq,-0.2869,-0.7946,-6.1015,186.6779,0.0000,True
5,ndmi_median,-0.2673,-0.3982,-0.0922,8.9156,0.0116,True
6,twi,-0.9992,-1.4196,-3.7202,131.4603,0.0000,True
7,sand_pct,0.6870,3.1586,14.0091,1413.2383,0.0000,True
8,clay_pct,0.1514,0.3003,0.3888,6.3658,0.0415,True
9,soc,-0.4612,-1.4360,-10.4109,281.4191,0.0000,True


In [17]:
b = po[[c for c in po.columns if c.startswith("b_ge_")]]
po2 = po.assign(spread=b.max(axis=1) - b.min(axis=1),
                mean_abs=b.abs().mean(axis=1))
po2["rel_spread"] = po2["spread"] / po2["mean_abs"].clip(lower=1e-6)
print(po2[["term","p","PO_violated","spread","rel_spread"]]
      .sort_values("rel_spread", ascending=False).round(3).to_string(index=False))
print("\nterms where slopes differ by >50% of their own magnitude:",
      int((po2["rel_spread"] > 0.5).sum()), "of", len(po2))

           term     p  PO_violated  spread  rel_spread
   temp_gs_mean 0.000         True   3.624       2.979
    aspect_east 0.205        False   0.230       2.909
   aspect_north 0.025         True   0.366       2.884
 ndmi_range_med 0.802        False   0.113       2.864
 rain_dryseason 0.000         True   1.702       2.844
        soil_ph 0.000         True   1.007       2.781
     soil_ph_sq 0.000         True  19.494       2.598
         soc_sq 0.000         True   7.336       2.569
     water_freq 0.000         True   0.260       2.458
            cec 0.000         True   0.656       2.448
temp_gs_mean_sq 0.000         True   5.815       2.429
            soc 0.000         True   9.950       2.425
      elevation 0.000         True   0.596       2.254
       sand_pct 0.000         True  13.322       2.238
            awc 0.000         True   0.684       2.114
         cec_sq 0.000         True   0.372       1.963
  solar_rad_iqr 0.712        False   0.130       1.950
          

In [18]:
import numpy as np, pandas as pd, statsmodels.api as sm
from sklearn.metrics import roc_auc_score, accuracy_score

codes = pd.Series(y_sel).cat.codes
print("per-split separability (labels are derived from these same features):\n")
for k in range(1, len(ORDER)):
    yb = (codes >= k).astype(int)
    r = sm.Logit(yb, sm.add_constant(X_sel[FINAL_COLS])).fit(disp=False, maxiter=200)
    p = r.predict(sm.add_constant(X_sel[FINAL_COLS]))
    print(f"  >= {ORDER[k]:3s}  n+={yb.sum():>6,}  AUC={roc_auc_score(yb,p):.4f}  "
          f"acc={accuracy_score(yb,(p>0.5).astype(int)):.4f}  "
          f"pseudo-R2={r.prsquared:.3f}  max|coef|={r.params[FINAL_COLS].abs().max():.1f}")

print("\nsplit coefficients for the largest spreads:")
big = po.reindex(po.assign(s=lambda d: d[[c for c in d.columns
      if c.startswith('b_ge_')]].max(1)-d[[c for c in d.columns
      if c.startswith('b_ge_')]].min(1)).sort_values("s", ascending=False).index)
print(big.head(8)[["term"] + [c for c in po.columns if c.startswith("b_ge_")]]
      .round(2).to_string(index=False))

per-split separability (labels are derived from these same features):

  >= S3   n+=43,236  AUC=0.9912  acc=0.9723  pseudo-R2=0.831  max|coef|=8.0
  >= S2   n+=28,236  AUC=0.9877  acc=0.9455  pseudo-R2=0.796  max|coef|=5.8
  >= S1   n+=13,236  AUC=0.9998  acc=0.9987  pseudo-R2=0.988  max|coef|=20.4

split coefficients for the largest spreads:
           term  b_ge_S3  b_ge_S2  b_ge_S1
     soil_ph_sq    -0.90    -1.21   -20.39
       sand_pct     0.69     3.16    14.01
            soc    -0.46    -1.44   -10.41
      slope_pct    -7.99    -5.83   -14.81
         soc_sq    -0.06    -1.12    -7.39
temp_gs_mean_sq    -0.29    -0.79    -6.10
   temp_gs_mean    -0.64    -0.03     2.98
            twi    -1.00    -1.42    -3.72


In [19]:
RULE = ["soil_ph","slope_pct","temp_gs_mean","soc","cec","sand_pct",
        "clay_pct","ndmi_median","twi","sunshine_rad"]
rule_cols = [c for v in RULE if v in GROUPS for c in GROUPS[v]]
free_cols = [c for c in cols_of(list(GROUPS)) if c not in rule_cols]

for name, cols in [("rule predictors only", rule_cols),
                   ("non-rule predictors only", free_cols),
                   ("all predictors", cols_of(list(GROUPS)))]:
    f, sd, k = cv_scores(cols)
    print(f"{name:26s} n={len(cols):3d}  macroF1={f:.3f} (sd {sd:.3f})  kappa={k:.3f}")

rule predictors only       n= 14  macroF1=0.860 (sd 0.003)  kappa=0.808
non-rule predictors only   n= 23  macroF1=0.517 (sd 0.068)  kappa=0.354
all predictors             n= 37  macroF1=0.856 (sd 0.007)  kappa=0.801


## Stage 8 — Final model on the full training set

Fitted on all ~300k training rows, with block-bootstrap confidence intervals (resampling
whole blocks, not cells, so spatial autocorrelation does not inflate precision).

**Watch the separation flag.** If it fires, the labels are near-deterministic functions
of the features — the circularity in deriving FAO classes from these same predictors,
showing up as a diagnostic.

In [20]:
tick("stage 8: final fit")
final = fit_ord(X_tr_full[FINAL_COLS], y_tr_full)

boot, blocks_tr = [], train[BLOCK]
ub = blocks_tr.unique()
for i in range(N_BOOT):
    bs  = rng.choice(ub, size=len(ub), replace=True)
    idx = np.flatnonzero(blocks_tr.isin(bs).to_numpy())
    if len(idx) > BOOT_N: idx = rng.choice(idx, BOOT_N, replace=False)
    try:
        yb = y_tr_full.iloc[idx].reset_index(drop=True)
        if pd.Series(yb).nunique() < len(ORDER): continue
        boot.append(fit_ord(X_tr_full.iloc[idx][FINAL_COLS].reset_index(drop=True),
                            yb).params[FINAL_COLS])
    except Exception:
        pass
    if (i+1) % 25 == 0: tick(f"  bootstrap {i+1}/{N_BOOT}")

boot = pd.DataFrame(boot)
coef = final.params[FINAL_COLS]
odds = pd.DataFrame({
    "coef": coef,
    "odds_ratio": np.exp(coef),
    "OR_lo95_block_boot": np.exp(boot.quantile(0.025)) if len(boot) else np.nan,
    "OR_hi95_block_boot": np.exp(boot.quantile(0.975)) if len(boot) else np.nan,
    "naive_p": final.pvalues[FINAL_COLS],     # optimistic under autocorrelation
    "separation_flag": coef.abs() > 8,
})
odds.to_csv(p_out("fs_final_odds_ratios.csv"))
if odds["separation_flag"].any():
    print("WARNING: possible separation -- labels may be near-deterministic "
          "functions of the features")
print(f"{len(boot)} of {N_BOOT} bootstrap fits converged")
odds.round(4)

[ 746.7 min] stage 8: final fit
[ 771.2 min]   bootstrap 25/100
[ 785.9 min]   bootstrap 50/100
[ 800.9 min]   bootstrap 75/100
[ 815.6 min]   bootstrap 100/100
100 of 100 bootstrap fits converged


,coef,odds_ratio,OR_lo95_block_boot,OR_hi95_block_boot,naive_p,separation_flag
soil_ph,0.1754,1.1917,1.0024,1.3385,0.0000,False
soil_ph_sq,-0.7425,0.4759,0.3917,0.5211,0.0000,False
slope_pct,-5.6813,0.0034,0.0025,0.0041,0.0000,False
temp_gs_mean,-0.5881,0.5554,0.4319,0.6541,0.0000,False
temp_gs_mean_sq,-0.2155,0.8061,0.6909,0.8800,0.0000,False
ndmi_median,-0.3822,0.6824,0.6399,0.7412,0.0000,False
twi,-0.8491,0.4278,0.3903,0.4726,0.0000,False
sand_pct,2.1340,8.4489,6.7485,10.1467,0.0000,False
clay_pct,0.1708,1.1863,1.0180,1.3186,0.0000,False
soc,-1.0677,0.3438,0.2846,0.4406,0.0000,False


## Stage 9 — Spatially blocked CV and the holdout

`StratifiedGroupKFold` keeps whole blocks together *and* balances classes across folds —
plain `GroupKFold` could hand you a fold with no S1 at all.

**The rows that matter most are the per-region holdout lines.** S1 is 95% Region I, so if
Region I scores well and Region II collapses, the model learned geography rather than
agronomy. That is a finding to report, not a bug to hide.

In [21]:
tick("stage 9")
def cv_scores(cols):
    f1s, ks = [], []
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for tr_i, va_i in splitter.split(X_sel, pd.Series(y_sel).astype(str), blk_sel):
        m = fit_ord(X_sel.iloc[tr_i][cols].reset_index(drop=True),
                    y_sel.iloc[tr_i].reset_index(drop=True))
        pred = np.asarray(m.predict(X_sel.iloc[va_i][cols].values)).argmax(axis=1)
        true = pd.Series(y_sel).cat.codes.to_numpy()[va_i]
        f1s.append(f1_score(true, pred, average="macro"))
        ks.append(cohen_kappa_score(true, pred))
    return float(np.mean(f1s)), float(np.std(f1s)), float(np.mean(ks))

dropped3   = {r["feature"] for r in log if r["decision"]=="dropped" and r["stage"].startswith("3")}
after_corr = [g for g in GROUPS if g not in dropped3]

rows = []
for name, cols in [("candidates after redundancy screen", cols_of(after_corr)),
                   ("final selected set", FINAL_COLS)]:
    f, fsd, k = cv_scores(cols)
    rows.append(dict(model=name, n_terms=len(cols), macroF1=f, macroF1_sd=fsd, kappa=k))

pred_te = np.asarray(final.predict(X_te[FINAL_COLS].values)).argmax(axis=1)
true_te = pd.Series(y_te).cat.codes.to_numpy()
rows.append(dict(model="final -> spatial holdout", n_terms=len(FINAL_COLS),
                 macroF1=f1_score(true_te, pred_te, average="macro"),
                 macroF1_sd=np.nan, kappa=cohen_kappa_score(true_te, pred_te)))
for reg in sorted(test[REGION].unique()):
    m_ = (test[REGION] == reg).to_numpy()
    rows.append(dict(model=f"final -> holdout Region {reg}", n_terms=len(FINAL_COLS),
                     macroF1=f1_score(true_te[m_], pred_te[m_], average="macro"),
                     macroF1_sd=np.nan, kappa=cohen_kappa_score(true_te[m_], pred_te[m_])))

cv = pd.DataFrame(rows)
cv.to_csv(p_out("fs_cv_comparison.csv"), index=False)
pd.DataFrame(log).to_csv(p_out("fs_selection_log.csv"), index=False)
tick("done")
cv.round(3)

[ 815.6 min] stage 9
[ 847.0 min] done


,model,n_terms,macroF1,macroF1_sd,kappa
0,candidates after redundancy screen,27,0.858,0.004,0.805
1,final selected set,23,0.858,0.004,0.806
2,final -> spatial holdout,23,0.837,NaN,0.774
3,final -> holdout Region I,23,0.830,NaN,0.755
4,final -> holdout Region II,23,0.789,NaN,0.783


### Holdout confusion matrix

In [22]:
cm = pd.DataFrame(confusion_matrix(true_te, pred_te, labels=range(len(ORDER))),
                  index=[f"true_{c}" for c in ORDER],
                  columns=[f"pred_{c}" for c in ORDER])
cm.to_csv(p_out("fs_holdout_confusion.csv"))
print("row-normalised (recall per class):")
display((cm.div(cm.sum(axis=1), axis=0) * 100).round(1))
cm

row-normalised (recall per class):


,pred_N,pred_S3,pred_S2,pred_S1
true_N,90.8,8.8,0.4,0.0
true_S3,11.1,76.0,12.8,0.0
true_S2,0.0,11.0,87.9,1.1
true_S1,0.0,0.0,23.3,76.7


,pred_N,pred_S3,pred_S2,pred_S1
true_N,21665,2092,102,0
true_S3,2706,18461,3121,7
true_S2,0,2424,19323,239
true_S1,0,0,439,1447


In [24]:
from sklearn.metrics import cohen_kappa_score
print("linear weighted  :", round(cohen_kappa_score(true_te, pred_te, weights="linear"), 3))
print("quadratic weighted:", round(cohen_kappa_score(true_te, pred_te, weights="quadratic"), 3))

linear weighted  : 0.834
quadratic weighted: 0.893


### Selection log — the audit trail for Section 3.4.3

In [25]:
sl = pd.DataFrame(log)
print(f"written to {OUT_DIR}")
sl[sl["stage"] != "1_transform"]

written to D:\Projects\capstone\fs_outputs


,feature,stage,decision,reason
31,stress_freq_med,3a_correlation,dropped,rho=-0.95 with ndmi_median; lower agronomic pr...
32,precip_total_iqr,3a_correlation,dropped,rho=0.90 with rain_dryseason; lower agronomic ...
33,ndvi_median,3a_correlation,dropped,rho=0.88 with ndmi_median; lower agronomic pri...
34,vpd_med,3a_correlation,dropped,rho=0.87 with temp_gs_mean; lower agronomic pr...
35,amplitude_med,3a_correlation,dropped,"rho=0.86 with ndmi_range_med; same priority, w..."
36,precip_cv_med,3a_correlation,dropped,rho=-0.85 with rain_dryseason; lower agronomic...
37,sunshine_rad,3a_correlation,dropped,"rho=-0.80 with rain_dryseason; same priority, ..."
38,total_n,3a_correlation,dropped,rho=-0.80 with temp_gs_mean; lower agronomic p...
39,dry_spell_max_med,3a_correlation,dropped,rho=-0.79 with rain_dryseason; lower agronomic...
40,soil_ph,4_screen,forced in,decisive factor (p=0)


map


In [26]:
import json, numpy as np
alphas = final.model.transform_threshold_params(
    final.params[len(FINAL_COLS):].to_numpy())

spec = {"final_cols": list(FINAL_COLS),
        "beta": {c: float(final.params[c]) for c in FINAL_COLS},
        "alphas": [float(a) for a in alphas],
        "order": ORDER, "vars": {}}

for v in active:                      # only the selected variables
    if v in CIRCULAR:
        spec["vars"][v] = {"kind": "circular", "cols": GROUPS[v]}
    elif v in OPTIMUM:
        c = OPTIMUM[v] if OPTIMUM[v] is not None else float(train[v].median())
        spec["vars"][v] = {"kind": "optimum", "center": float(c),
                           "std": float(train[v].std()), "cols": GROUPS[v]}
    else:
        s = np.log1p(train[v].clip(lower=0)) if v in SKEWED else train[v].astype(float)
        spec["vars"][v] = {"kind": "scale", "log1p": v in SKEWED,
                           "mean": float(s.mean()), "std": float(s.std()),
                           "cols": GROUPS[v]}

json.dump(spec, open(r"D:\Projects\capstone\model_spec.json", "w"), indent=1)
print(f"saved {len(FINAL_COLS)} terms, {len(spec['vars'])} variables")

saved 23 terms, 18 variables


In [28]:
%run "D:\Projects\capstone\predict_and_map.py"

94 parts | 23 model terms
  reading 25 source columns
  10/94  3,540,148 cells (0.2 min)
  20/94  6,519,396 cells (0.3 min)
  30/94  9,938,499 cells (0.5 min)
  50/94  22,082,310 cells (1.1 min)
  60/94  28,321,985 cells (1.4 min)
  70/94  29,931,607 cells (1.4 min)
  80/94  32,579,699 cells (1.6 min)
  90/94  39,037,542 cells (1.9 min)
  94/94  41,592,438 cells (2.0 min)

rendering

WALL-TO-WALL PREDICTION
  cells predicted : 41,592,438
  exact agreement : 86.97%
  within one class: 99.91%

confusion (rows = labelled class):
           pred_N  pred_S3  pred_S2  pred_S1
true_N   21423743  1589953    28021      393
true_S3    987311  9856825  2052535    10792
true_S2         3   650023  4875971    98793
true_S1         0        0     3324    14751

class shares:
  N   labelled  55.40%   predicted  53.88%   area 20,738 km2
  S3  labelled  31.03%   predicted  29.08%   area 11,617 km2
  S2  labelled  13.52%   predicted  16.73%   area 5,062 km2
  S1  labelled   0.04%   predicted   0.30%   a